# Bangla Licence Plate Datasets

Collects the datasets into `/kaggle/working/bangladeshi-lpr/`, verifies them, and prepares them to be saved as one private Kaggle dataset for training.

| Dataset | Content | Folder |
|---|---|---|
| [Bangladeshi License Plate Recognition](https://www.kaggle.com/datasets/syednahinhossain/bangladeshi-license-plate-recognition-dataset) | Full frames with plate boxes; plate crops with character boxes | `bd-lpr/` |
| [Bangla License Plate (images and videos)](https://tinyurl.com/p7e8tszk) | Full frames of vehicles; traffic videos | `bd-lpr-dhaka/` |
| [BLPD](https://github.com/prosensarker4/BLPD) | Full frames with plate boxes; plate crops with character boxes (YOLO) | `bd-lpr-blpd/` |
| Roboflow Universe ([1](https://universe.roboflow.com/bangla-license-plate-detection-by-yolov8/bangla-license-plate-detection-dkqe3), [2](https://universe.roboflow.com/bangla-car-license-plate-detection/bangla-license-plate-detection-p3xyb)) | Full frames with plate boxes (YOLO) | `roboflow-*/` |

**Setup (Kaggle notebook)**
1. Add the Kaggle dataset above as an input (**Add Input**).
2. For the Roboflow datasets, add a free Roboflow API key as the secret `ROBOFLOW_API_KEY` (Add-ons → Secrets). Without it, this step is skipped.

In [ ]:
%pip install -q gdown roboflow

## Setup

In [ ]:
import json
import random
import shutil
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

INPUT = Path("/kaggle/input")
DATA = Path("/kaggle/working/bangladeshi-lpr")
DATA.mkdir(parents=True, exist_ok=True)

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".3gp"}
PLATE_LABELS = {"license_plate", "licence_plate", "plate"}


def is_downloaded(folder: Path) -> bool:
    return folder.exists() and any(p.is_file() for p in folder.rglob("*"))


def extract_archives(folder: Path) -> None:
    for archive in list(folder.rglob("*.zip")):
        target = archive.with_suffix("")
        if not target.exists():
            with zipfile.ZipFile(archive) as z:
                z.extractall(target)
        archive.unlink()


def count_files(folder: Path, extensions: set) -> int:
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in extensions)

## Kaggle input

In [ ]:
def copy_input(slug: str, name: str) -> Path:
    source = next((p for p in INPUT.rglob(slug) if p.is_dir()), None)
    if source is None:
        raise FileNotFoundError(f"Add the '{slug}' dataset as an input.")
    target = DATA / name
    if not is_downloaded(target):
        shutil.copytree(source, target, dirs_exist_ok=True)
    extract_archives(target)
    return target


bd_lpr = copy_input("bangladeshi-license-plate-recognition-dataset", "bd-lpr")

## Google Drive

If videos are missing after verification, download the Drive folder manually and add it to the saved dataset.

In [ ]:
import gdown

bd_lpr_dhaka = DATA / "bd-lpr-dhaka"
if not is_downloaded(bd_lpr_dhaka):
    bd_lpr_dhaka.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(
        "https://drive.google.com/drive/folders/1751-4Uei0rXW9LCk_yzr9rqXzflsJG_Y",
        output=str(bd_lpr_dhaka),
        remaining_ok=True,
    )
extract_archives(bd_lpr_dhaka)

## GitHub

In [ ]:
import subprocess

bd_lpr_blpd = DATA / "bd-lpr-blpd"
if not is_downloaded(bd_lpr_blpd):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/prosensarker4/BLPD.git", str(bd_lpr_blpd)], check=True)
    shutil.rmtree(bd_lpr_blpd / ".git", ignore_errors=True)

## Roboflow Universe

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    roboflow_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    roboflow_key = None

if roboflow_key:
    from roboflow import Roboflow

    rf = Roboflow(api_key=roboflow_key)
    projects = {
        "roboflow-dkqe3": ("bangla-license-plate-detection-by-yolov8", "bangla-license-plate-detection-dkqe3"),
        "roboflow-p3xyb": ("bangla-car-license-plate-detection", "bangla-license-plate-detection-p3xyb"),
    }
    for name, (workspace, project_id) in projects.items():
        folder = DATA / name
        if is_downloaded(folder):
            continue
        project = rf.workspace(workspace).project(project_id)
        latest = max(int(v["id"].split("/")[-1]) for v in project.get_version_information())
        project.version(latest).download("yolov8", location=str(folder))
else:
    print("ROBOFLOW_API_KEY not set; Roboflow datasets skipped.")

## Verification

In [ ]:
def label_names(xml_path: Path) -> set:
    try:
        return {obj.findtext("name") for obj in ET.parse(xml_path).getroot().iter("object")}
    except ET.ParseError:
        return set()


bd_lpr_xml = {p: label_names(p) for p in bd_lpr.rglob("*.xml")}
localization_xml = [p for p, names in bd_lpr_xml.items() if names and names <= PLATE_LABELS]
character_xml = [p for p, names in bd_lpr_xml.items() if names and not names <= PLATE_LABELS]

checks = [
    ("bd-lpr: localization labels", len(localization_xml), 4795),
    ("bd-lpr: character labels", len(character_xml), 4061),
    ("bd-lpr: images", count_files(bd_lpr, IMAGE_EXT), 6517 + 4061),
    ("bd-lpr-dhaka: images", count_files(bd_lpr_dhaka, IMAGE_EXT), 1000),
    ("bd-lpr-dhaka: videos", count_files(bd_lpr_dhaka, VIDEO_EXT), 79),
    ("bd-lpr-blpd: detection images", count_files(bd_lpr_blpd / "LPD_Dataset", IMAGE_EXT), 2140),
    ("bd-lpr-blpd: character images", count_files(bd_lpr_blpd / "LPNR_Dataset", IMAGE_EXT), 7217),
]
checks += [(f"{p.name}: images", count_files(p, IMAGE_EXT), 1) for p in sorted(DATA.glob("roboflow-*"))]

print(f"{'Part':32} {'Found':>8} {'Expected':>9}  Status")
for name, found, expected in checks:
    print(f"{name:32} {found:8d} {expected:9d}  {'OK' if found >= expected else 'MISSING'}")

inventory = [{"part": n, "found": f, "expected": e} for n, f, e in checks]
(DATA / "inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")

## Samples

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image


def read_boxes(label_path: Path, width: int, height: int):
    if label_path.suffix == ".xml":
        for obj in ET.parse(label_path).getroot().iter("object"):
            b = obj.find("bndbox")
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            yield obj.findtext("name"), x1, y1, x2 - x1, y2 - y1
    else:
        for line in label_path.read_text(errors="ignore").splitlines():
            parts = line.split()
            if len(parts) == 5:
                c, xc, yc, w, h = parts[0], *map(float, parts[1:])
                yield c, (xc - w / 2) * width, (yc - h / 2) * height, w * width, h * height


def show_samples(title: str, root: Path, labels: list, n: int = 5):
    images = {p.stem: p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXT}
    pairs = [(images[l.stem], l) for l in labels if l.stem in images]
    if not pairs:
        print(f"{title}: no labelled samples found")
        return
    random.seed(0)
    pairs = random.sample(pairs, min(n, len(pairs)))
    fig, axes = plt.subplots(1, len(pairs), figsize=(4 * len(pairs), 4), squeeze=False)
    for ax, (image_path, label_path) in zip(axes[0], pairs):
        image = Image.open(image_path).convert("RGB")
        ax.imshow(image)
        for name, x, y, w, h in read_boxes(label_path, *image.size):
            ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color="lime", lw=2))
            ax.text(x, y, name, color="yellow", fontsize=7, va="bottom")
        ax.set_title(f"{image.size[0]}x{image.size[1]}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def labels_in(folder: Path) -> list:
    return [p for p in folder.rglob("*") if p.suffix in {".xml", ".txt"} and p.name not in {"README.txt", "classes.txt"}]


show_samples("bd-lpr: localization", bd_lpr, localization_xml)
show_samples("bd-lpr: characters", bd_lpr, character_xml)
show_samples("bd-lpr-dhaka: images", bd_lpr_dhaka, labels_in(bd_lpr_dhaka))
show_samples("bd-lpr-blpd: detection", bd_lpr_blpd / "LPD_Dataset", labels_in(bd_lpr_blpd / "LPD_Dataset"))
show_samples("bd-lpr-blpd: characters", bd_lpr_blpd / "LPNR_Dataset", labels_in(bd_lpr_blpd / "LPNR_Dataset"))
for folder in sorted(DATA.glob("roboflow-*")):
    show_samples(folder.name, folder, labels_in(folder))

videos = sorted(p for p in bd_lpr_dhaka.rglob("*") if p.suffix.lower() in VIDEO_EXT)
if videos:
    ok, frame = cv2.VideoCapture(str(videos[0])).read()
    if ok:
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"bd-lpr-dhaka: {videos[0].name}", fontsize=9)
        plt.axis("off")
        plt.show()